In [5]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
import joblib
import os

In [7]:
df = pd.read_csv('../data/processed/featured_data.csv')

print(f"Shape: {df.shape}")
print(f"Fraud ratio: {df['isFraud'].mean():.4f}")
print(f"Missing values: {df.isnull().sum().sum()}")

Shape: (590540, 235)
Fraud ratio: 0.0350
Missing values: 0


In [8]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train fraud ratio: {y_train.mean():.4f}")

Train shape: (472432, 234)
Test shape: (118108, 234)
Train fraud ratio: 0.0350


In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {y_train_balanced.value_counts().to_dict()}")

Before SMOTE: {0: 455902, 1: 16530}
After SMOTE: {0: 455902, 1: 455902}


In [10]:
os.makedirs('../models', exist_ok=True)
mlflow.set_experiment("fraud-detection")

with mlflow.start_run():

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='auc',
        early_stopping_rounds=20
    )

    model.fit(
        X_train_balanced, y_train_balanced,
        eval_set=[(X_test, y_test)],
        verbose=50
    )

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    auc       = roc_auc_score(y_test, y_prob)
    f1        = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    mlflow.xgboost.log_model(model, "model")

    print(f"AUC:       {auc:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

2026/03/18 17:31:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/18 17:31:44 INFO mlflow.store.db.utils: Updating database tables
2026/03/18 17:31:44 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


[0]	validation_0-auc:0.83246
[50]	validation_0-auc:0.86637
[100]	validation_0-auc:0.88103
[150]	validation_0-auc:0.89129
[200]	validation_0-auc:0.89932
[250]	validation_0-auc:0.90515
[299]	validation_0-auc:0.91016


2026/03/18 17:33:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


AUC:       0.9102
F1:        0.5562
Precision: 0.7842
Recall:    0.4309


In [11]:
joblib.dump(model, '../models/fraud_model.pkl')
print("Model saved to models/fraud_model.pkl")

Model saved to models/fraud_model.pkl


In [12]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    print(f"Threshold {t} → F1: {f1:.3f} Precision: {precision:.3f} Recall: {recall:.3f}")

Threshold 0.5 → F1: 0.556 Precision: 0.784 Recall: 0.431
Threshold 0.4 → F1: 0.571 Precision: 0.703 Recall: 0.481
Threshold 0.3 → F1: 0.566 Precision: 0.594 Recall: 0.541
Threshold 0.2 → F1: 0.520 Precision: 0.450 Recall: 0.616
Threshold 0.1 → F1: 0.394 Precision: 0.269 Recall: 0.729


In [13]:
import joblib

joblib.dump(model, '../models/fraud_model.pkl')
joblib.dump(0.4, '../models/threshold.pkl')

print("Model and threshold saved")

Model and threshold saved
